In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve() / "src"))
from weedvlm.evaluation import scoring
from weedvlm.evaluation import bboxes
import yaml

import pandas as pd
import sqlite3
import json

import seaborn as sns
import matplotlib.pyplot as plt

conn = sqlite3.connect("./../questions/benchmark.db")
questions_df = pd.read_sql_query("SELECT * FROM questions", conn)
responses_df = pd.read_sql_query("SELECT * FROM responses WHERE status = 'ok'", conn)

## Scoring for Weed Species ID

This sections performs the scoring for Weed species identification, for both the multiple choice and open ended questions. This only considers the results for a particular set of runs, and will create a comparison across species. The overall result will be a comparison between species.

In [2]:
species_id_questions = questions_df[questions_df["task"] == "species_id"]
species_id_responses = responses_df.merge(
    species_id_questions.rename(columns={"id": "question_id"}),
    on="question_id",
    how="inner"
)
# species_id_responses["match"] = (species_id_responses.answer.str.lower() == species_id_responses.expected_answer.str.lower())
aliases = scoring.load_aliases(yaml.safe_load(Path("./../species-aliases.yaml").read_text(encoding="utf-8")))
species_id_responses["match"] = species_id_responses.apply(
    lambda row: scoring.open_ended_correct(row["answer"], row["expected_answer"], aliases),
    axis=1,
)
wsi_scores = (
    species_id_responses
    .groupby(["run", "question_type"])["match"]
    .mean()
    .reset_index()
)

In [3]:
# Get the overall accuracy
overall_accuracy = species_id_responses.match.sum() / species_id_responses["match"].count()
# Get the MCQ only accuracy
mcq_responses = species_id_responses[species_id_responses.question_type == "species_id_multiple_choice"]
mcq_accuracy = mcq_responses.match.sum() / mcq_responses["match"].count()
# Get the Open-ended question accuracy
open_ended_responses = species_id_responses[species_id_responses.question_type == "species_id_open_ended"]
open_ended_accuracy = open_ended_responses.match.sum() / open_ended_responses["match"].count()

print(f"Overall accuracy: {overall_accuracy}, MCQ accuracy: {mcq_accuracy}, Open-ended accuracy: {open_ended_accuracy}")
# Plot this in seaborn

Overall accuracy: 0.770210154931738, MCQ accuracy: 0.8651085756348914, Open-ended accuracy: 0.29493087557603687


### Per species breakdown

Where does it fail on a per-species level?

In [4]:
print(species_id_responses.groupby(["run", "benchmark_class", "question_type"])["match"].mean().unstack("question_type"))

question_type                                  species_id_multiple_choice  \
run                   benchmark_class                                       
claude-haiku-full-run Amaranthus tuberculatus                    0.375000   
                      Ambrosia artemisiifolia                    0.936709   
                      Chenopodium album                          0.625000   
                      Cockspur grass                             0.800000   
                      Creeping thistle                           0.675000   
...                                                                   ...   
gpt-6-luna-full-run   Sida rhombifolia                           0.812500   
                      Taraxacum officinale                       0.712500   
                      Thorn apple                                0.950000   
                      White goosefoot                            0.887500   
                      Xanthium strumarium                        0.962500   

## Fine-grained identification

Now we will look at fine-grained identification. Lets get this setup.

In [5]:
fg_questions = questions_df[questions_df["task"] == "fine_grained"]
fg_responses = responses_df.merge(
    fg_questions.rename(columns={"id": "question_id"}),
    on="question_id",
    how="inner"
)
fg_responses["match"] = fg_responses.answer == fg_responses.expected_answer
fg_accuracy = fg_responses["match"].mean()
print(f"Overall fine-grained accuracy: {fg_accuracy}")
# Per species breakdown
print(fg_responses.groupby(["run", "benchmark_class"])["match"].mean())
print(fg_responses.groupby("run")["match"].mean())

Overall fine-grained accuracy: 0.7555555555555555
run                          benchmark_class  
claude-haiku-full-run        different_species    0.620
                             same_species         0.755
deepseek-4-1-flash-full-run  different_species    0.640
                             same_species         0.845
gemma4-31b-full-run          different_species    0.710
                             same_species         0.875
gpt-6-1-sol-partial-run      different_species    0.700
                             same_species         0.800
gpt-6-luna-full-run          different_species    0.790
                             same_species         0.815
Name: match, dtype: float64
run
claude-haiku-full-run          0.6875
deepseek-4-1-flash-full-run    0.7425
gemma4-31b-full-run            0.7925
gpt-6-1-sol-partial-run        0.7500
gpt-6-luna-full-run            0.8025
Name: match, dtype: float64


## Grounded Weed VQA

Let's now look at how well the VLM is able to do grounded weed VQA.

In [6]:
gwvqa_df = pd.read_sql_query(
    """
    SELECT q.id AS question_id, q.question_type, q.benchmark_class, q.expected_answer,
           (i.path LIKE '%\\_gvqa\\_crop%' ESCAPE '\\') AS is_crop_baseline
    FROM questions q
    JOIN images i on i.id = q.image_id
    WHERE q.task = 'grounded_vqa'
    """,
    conn
)
gwvqa_questions = questions_df[questions_df["task"] == "grounded_vqa"]
gwvqa_responses = responses_df.merge(
    gwvqa_df,
    on="question_id",
    how="inner"
)
gwvqa_responses = gwvqa_responses[gwvqa_responses.is_crop_baseline == 0]
gwvqa_responses["match"] = gwvqa_responses.answer == gwvqa_responses.expected_answer
gwvqa_accuracy = gwvqa_responses.groupby("run")["match"].mean()
print(gwvqa_accuracy)
# Per species breakdown
print(gwvqa_responses.groupby(["run", "benchmark_class"])["match"].mean())

run
claude-haiku-full-run          0.946667
deepseek-4-1-flash-full-run    0.936455
gemma4-31b-full-run            0.860000
gpt-6-1-sol-partial-run        0.966667
gpt-6-luna-full-run            0.910000
Name: match, dtype: float64
run                    benchmark_class        
claude-haiku-full-run  Amaranthus tuberculatus    0.85
                       Ambrosia artemisiifolia    1.00
                       Cockspur grass             0.90
                       Creeping thistle           0.90
                       Eclipta                    0.90
                                                  ... 
gpt-6-luna-full-run    Ipomoea indica             0.90
                       Mollugo verticillata       0.85
                       Portulaca oleracea         0.90
                       Red root amaranth          0.95
                       Sida rhombifolia           0.95
Name: match, Length: 75, dtype: float64


## Weed localisation

Let's now look at weed localisation

In [7]:
wl_questions = questions_df[questions_df.task == "localisation"]
wl_responses = responses_df.merge(
    wl_questions.rename(columns={"id": "question_id"}),
    on="question_id",
    how="inner"
)
wl_responses.head()

def score_row(row, bbox_format=None):
    parsed = json.loads(row["parsed"])
    fmt = (bboxes.parse_bbox_format(bbox_format) if bbox_format
           else bboxes.infer_bbox_format(parsed.get("response_format")))
    try:
        pred, used = bboxes.to_pixel_boxes(parsed.get("bboxes"), fmt, row["image_width"], row["image_height"])
    except ValueError as e:
        return pd.Series({"outcome": "invalid", "error": str(e), "localisation_score": 0.0})
    return pd.Series({"outcome": "scored", "bbox_format": str(used), "pred_boxes": pred, **bboxes.score_boxes(json.loads(row["boxes"]), pred)})

scored = pd.concat([wl_responses[["id", "run", "model", "question_id", "benchmark_class"]], wl_responses.apply(score_row, axis=1)], axis=1)
print(scored.groupby("run")[["localisation_score", "matched_iou"]].mean())

                             localisation_score  matched_iou
run                                                         
claude-haiku-full-run                  0.336071     0.689142
deepseek-4-1-flash-full-run            0.249398     0.485870
gemma4-31b-full-run                    0.384303     0.733607
gpt-6-1-sol-partial-run                0.327589     0.644631
gpt-6-luna-full-run                    0.394123     0.623609


In [8]:
# Work out whether we fit too many bounding boxes
wl_responses.head()

,id,run,model,question_id,status,answer,reasoning,parsed,raw_response,warnings,...,expected_answer,choices,answer_index,ablation_of,true_weed_count,true_weed_coverage_fraction,target_species,image_width,image_height,boxes
0,2533,claude-haiku-full-run,anthropic/claude-haiku-5.5,CropAndWeed:4523:hybrid goosefoot:localisation,ok,"[[618, 265, 1370, 803]]",The single plant in the center is a goosefoot-...,"{""bboxes"": [[618, 265, 1370, 803]], ""response_...","{""bboxes"": [[618, 265, 1370, 803]], ""response_...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,Hybrid goosefoot,1920.0,1088.0,"[[616.0, 265.0, 753.0, 542.0]]"
1,2538,claude-haiku-full-run,anthropic/claude-haiku-5.5,CropAndWeed:4518:hybrid goosefoot:localisation,ok,"[[628, 327, 1118, 897], [0, 165, 352, 612]]","Both boxes cover plants with broad, triangular...","{""bboxes"": [[628, 327, 1118, 897], [0, 165, 35...","{""bboxes"": [[628, 327, 1118, 897], [0, 165, 35...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,Hybrid goosefoot,1920.0,1088.0,"[[627.0, 324.0, 493.0, 579.0]]"
2,2539,claude-haiku-full-run,anthropic/claude-haiku-5.5,CropAndWeed:4529:hybrid goosefoot:localisation,ok,"[[1595, 470, 1920, 935]]",The large plant at the bottom right has the br...,"{""bboxes"": [[1595, 470, 1920, 935]], ""response...","{""bboxes"": [[1595, 470, 1920, 935]], ""response...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,Hybrid goosefoot,1920.0,1088.0,"[[1594.0, 467.0, 326.0, 470.0]]"
3,2542,claude-haiku-full-run,anthropic/claude-haiku-5.5,CropAndWeed:4560:hybrid goosefoot:localisation,ok,"[[800, 258, 1495, 935], [330, 365, 780, 815], ...",I identified the goosefoot plants by their tri...,"{""bboxes"": [[800, 258, 1495, 935], [330, 365, ...","{""bboxes"": [[800, 258, 1495, 935], [330, 365, ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,Hybrid goosefoot,1920.0,1088.0,"[[362.0, 443.0, 428.0, 377.0], [796.0, 252.0, ..."
4,2544,claude-haiku-full-run,anthropic/claude-haiku-5.5,CropAndWeed:3124:hybrid goosefoot:localisation,ok,"[[730, 378, 1235, 805], [1305, 768, 1545, 1088]]","The large central plant with angular, maple-li...","{""bboxes"": [[730, 378, 1235, 805], [1305, 768,...","{""bboxes"": [[730, 378, 1235, 805], [1305, 768,...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,Hybrid goosefoot,1920.0,1088.0,"[[1307.0, 756.0, 240.0, 332.0], [731.0, 373.0,..."


In [9]:
species_id_responses.question_type.unique()
questions_df["task"].unique()

<ArrowStringArray>
['species_id', 'fine_grained', 'density', 'grounded_vqa', 'localisation']
Length: 5, dtype: str

## Weed Density estimation

Finally lets look at weed density estimation

In [10]:
wde_questions = questions_df[questions_df.task == "density"]
wde_responses = responses_df.merge(
    wde_questions.rename(columns={"id": "question_id"}),
    on="question_id",
    how="inner"
)
wde_responses["match"] = wde_responses.answer.str.lower() == wde_responses.expected_answer.str.lower()
wde_accuracy = wde_responses.groupby("run")["match"].mean()
print(wde_accuracy)
# Per species breakdown
print(wde_responses.groupby(["run", "benchmark_class"])["match"].mean())
parsed = wde_responses.parsed.apply(json.loads)
wde_responses["est_count"] = parsed.str["estimated_weed_count"] # type: ignore
wde_responses["est_coverage_pct"] = parsed.str["estimated_weed_coverage_percent"] # type: ignore
wde_responses["count_error"] = wde_responses.est_count - wde_responses.true_weed_count
wde_responses["coverage_error_pp"] = (
    wde_responses.est_coverage_pct - wde_responses.true_weed_coverage_fraction * 100
)
# Compute the MAE for each of these 
print(wde_responses.groupby("run")["count_error"].agg(lambda s: s.abs().mean())) # MAE
print(wde_responses.groupby("run")["count_error"].mean()) # Bias
# Now for the coverage error

print(wde_responses.groupby("run")["coverage_error_pp"].agg(lambda s: s.abs().mean())) # MAE
print(wde_responses.groupby("run")["coverage_error_pp"].mean()) # Bias

run
claude-haiku-full-run          0.622785
deepseek-4-1-flash-full-run    0.445000
gemma4-31b-full-run            0.415000
gpt-6-1-sol-partial-run        0.519608
gpt-6-luna-full-run            0.415000
Name: match, dtype: float64
run                          benchmark_class
claude-haiku-full-run        high               0.535354
                             low                0.939394
                             medium             0.752577
                             none               0.270000
deepseek-4-1-flash-full-run  high               0.200000
                             low                0.960000
                             medium             0.350000
                             none               0.270000
gemma4-31b-full-run          high               0.070000
                             low                0.940000
                             medium             0.300000
                             none               0.350000
gpt-6-1-sol-partial-run      high      

In [11]:
print(responses_df.columns)

Index(['id', 'run', 'model', 'question_id', 'status', 'answer', 'reasoning',
       'parsed', 'raw_response', 'warnings', 'error_kind', 'error_message',
       'error_http_status', 'attempts', 'retry_log', 'finish_reason',
       'latency_seconds', 'usage', 'total_tokens', 'cost_usd', 'generation_id',
       'provider', 'image_sha256', 'prompt_version', 'created_at'],
      dtype='str')


In [14]:
# Save the dataframe for later use.

# ============================================================
# Build unified response-level scored dataframe
# ============================================================

# Start from every successfully evaluated response + question metadata
analysis_df = responses_df.merge(
    questions_df.rename(columns={"id": "question_id"}),
    on="question_id",
    how="inner",
    suffixes=("_response", "_question"),
)

# ------------------------------------------------------------
# 1. Categorical correctness
# ------------------------------------------------------------

match_scores = pd.concat([
    species_id_responses[["id", "match"]],
    fg_responses[["id", "match"]],
    gwvqa_responses[["id", "match"]],
    wde_responses[["id", "match"]],
], ignore_index=True)

# Each response should only appear once
match_scores = match_scores.drop_duplicates(subset="id")

analysis_df = analysis_df.merge(
    match_scores,
    on="id",
    how="left",
)

# ------------------------------------------------------------
# 2. Localisation metrics
# ------------------------------------------------------------

# IMPORTANT:
# Add "id" to the columns retained when constructing `scored`
# in your localisation section.

localisation_metrics = scored[
    [
        "id",
        "outcome",
        "localisation_score",
        "matched_iou",
        "bbox_format",
        "pred_boxes",
    ]
].copy()

# Store as JSON text
localisation_metrics["pred_boxes"] = localisation_metrics["pred_boxes"].apply(
    lambda b: json.dumps(b) if isinstance(b, list) else None
)

analysis_df = analysis_df.merge(
    localisation_metrics,
    on="id",
    how="left",
)

# ------------------------------------------------------------
# 3. Density metrics
# ------------------------------------------------------------

wde_responses["abs_count_error"] = (
    wde_responses["count_error"].abs()
)

wde_responses["abs_coverage_error_pp"] = (
    wde_responses["coverage_error_pp"].abs()
)

density_metrics = wde_responses[
    [
        "id",
        "est_count",
        "est_coverage_pct",
        "count_error",
        "abs_count_error",
        "coverage_error_pp",
        "abs_coverage_error_pp",
    ]
].copy()

analysis_df = analysis_df.merge(
    density_metrics,
    on="id",
    how="left",
)

# ------------------------------------------------------------
# 4. Save for downstream analysis
# ------------------------------------------------------------

analysis_df.to_parquet(
    "weedvlm_scored_responses.parquet",
    index=False,
)

print(analysis_df.shape)
display(analysis_df.head())

(12560, 57)


,id,run,model,question_id,status,answer,reasoning,parsed,raw_response,warnings,...,localisation_score,matched_iou,bbox_format,pred_boxes,est_count,est_coverage_pct,count_error,abs_count_error,coverage_error_pp,abs_coverage_error_pp
0,1,claude-haiku-full-run,anthropic/claude-haiku-5.5,CottonWeedDet12:1910:portulaca oleracea:specie...,ok,Portulaca oleracea,"The boxed seedlings have thick, fleshy, spoon-...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,claude-haiku-full-run,anthropic/claude-haiku-5.5,CottonWeedDet12:1679:portulaca oleracea:specie...,ok,Portulaca oleracea,"The seedling has succulent, spatulate to obova...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,claude-haiku-full-run,anthropic/claude-haiku-5.5,CottonWeedDet12:1522:portulaca oleracea:specie...,ok,Portulaca oleracea,"The boxed plants have fleshy, spatulate to obo...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,claude-haiku-full-run,anthropic/claude-haiku-5.5,CottonWeedDet12:1428:portulaca oleracea:specie...,ok,Portulaca oleracea,"The plant has fleshy, spatulate to obovate lea...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,claude-haiku-full-run,anthropic/claude-haiku-5.5,CottonWeedDet12:1557:portulaca oleracea:specie...,ok,Portulaca oleracea,"The plant has fleshy, spatulate to paddle-shap...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...","{""answer"": ""Portulaca oleracea"", ""reasoning"": ...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
